# Somo la 10 - Mawakala ya AI katika Uzalishaji

Katika somo hili utajifunza **mitindo ya uzalishaji** kwa mawakala wa AI ukitumia Microsoft Agent Framework (Python). Tutafunika:

- **Ufuatiliaji** — kuongeza upimaji wa muda na uandishi wa kumbukumbu kwenye mwingiliano wa wakala
- **Tathmini** — kutumia wakala mtathmini kupima ubora wa majibu
- **Usimamizi wa gharama** — mikakati ya uboreshaji wa tokeni na uchaguzi wa modeli

Senario ni **mwakala wa usafiri** ambaye husaidia watumiaji kupanga safari, na ufuatiliaji na tathmini vimewekwa juu yake.


## Usanidi


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Mambo ya Kuzingatia katika Uzalishaji

Kusogeza mawakala wa AI kutoka kwa prototaipu kwenda uzalishaji kunahitaji umakini makini kwa nguzo tatu:

1. **Ufuatiliaji** — Unahitaji uonekano wa kile ambacho wakala anafanya, muda gani inachukua, na ni zana gani anazozitumia. Bila ufuatiliaji na kurekodi, kutambua na kurekebisha matatizo kwenye uzalishaji karibu haiwezekani.

2. **Tathmini** — Ukaguzi wa ubora kiotomatiki unahakikisha majibu ya wakala yanabaki sahihi, kamili, na ya msaada kwa muda. Wakala wa kutathmini anaweza kutoa alama kwa majibu kwa mujibu wa vigezo vilivyowekwa.

3. **Usimamizi wa Gharama** — Matumizi ya tokeni yanaathiri gharama moja kwa moja. Mikakati kama uboreshaji wa maelekezo, uchaguzi wa modeli, na uhifadhi wa muda husaidia kudhibiti matumizi bila kuhatarisha ubora.


## Kujenga Wakala Anayeweza Kufuatiliwa

Tunabainisha zana za usafiri na kuwekea wito la wakala kipimo cha muda ili tuweze kufuatilia ucheleweshaji. Katika uzalishaji, ungeunganisha na OpenTelemetry au backend ya ufuatiliaji inayofanana.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Mifumo ya Tathmini

Mfumo wa kawaida wa uzalishaji ni kutumia wakala wa pili kama **mtathmini**. Mtathmini hutoa alama kwa jibu la wakala mkuu dhidi ya vigezo vilivyowekwa kabla kama ukamilifu, usahihi, na uwezo wa kusaidia.

Hii inawezesha:
- Vizingiti vya ubora vya kiotomatiki kabla majibu hayafikie watumiaji
- Ugundaji wa kurudi nyuma wakati maelekezo au mifano zinapobadilika
- Ufuatiliaji endelevu wa utendaji wa wakala kwa muda


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Mikakati ya Usimamizi wa Gharama

Kudhibiti gharama ni muhimu kwa maajenti wa AI katika uzalishaji. Hapa kuna mikakati muhimu:

| Mkakati | Maelezo |
|---|---|
| **Uboreshaji wa maagizo (prompt)** | Weka maelekezo ya mfumo mafupi. Ondoa muktadha rudufu ili kupunguza tokeni za ingizo. |
| **Uchaguzi wa modeli** | Tumia modeli ndogo, nafuu (mfano GPT-4o-mini) kwa kazi rahisi kama uainishaji au uchimbaji, na wahifadhi modeli kubwa kwa ajili ya uamuzi wa ugumu zaidi. |
| **Kuweka kwenye cache** | Hifadhi matokeo ya zana na maswali yanayotokea mara kwa mara ili kuepuka wito wa API yanayojirudia. |
| **Bajeti za tokeni** | Weka mipaka ya `max_tokens` ili kuzuia majibu yasiyotarajiwa kuwa marefu. |
| **Kuweka kwa makundi** | Unganisha maswali mengi ya watumiaji katika wito mmoja wa API inapowezekana. |

Kivitendo, mbinu ya tabaka inafanya kazi vizuri: elekeza maombi rahisi kwa modeli ya haraka na nafuu, na tuma tu maswali yenye ugumu mkubwa kwa modeli yenye uwezo zaidi.


## Muhtasari

Katika somo hili ulijifunza jinsi ya:

1. **Ongeza ufuatiliaji** kwa mwingiliano wa wakala kwa kupima muda na kuandika kumbukumbu, ukiweka msingi wa kufuatilia na uangalizi.
2. **Tathmini majibu ya wakala** kiotomatiki kwa kutumia wakala mtathmini anayetoa alama kwa ukamilifu, usahihi, na ufanisi.
3. **Dhibiti gharama** kupitia uboreshaji wa prompti, uchaguzi wa modeli, caching, na bajeti za tokeni.

Mifano hii ya uzalishaji husaidia kuhakikisha kwamba mawakala wako wa AI ni wa kuaminika, wanaweza kupimika, na wana gharama nafuu kwa kiwango kikubwa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Kauli ya kutokuwajibika:
Nyaraka hii imetafsiriwa kwa kutumia huduma ya tafsiri ya AI Co-op Translator (https://github.com/Azure/co-op-translator). Ingawa tunajitahidi kufanikiwa kwa usahihi, tafadhali kumbuka kwamba tafsiri za kiotomatiki zinaweza kuwa na makosa au kasahihi zisizokamilika. Nyaraka ya asili kwa lugha yake ya asili inapaswa kutambulika kama chanzo chenye mamlaka. Kwa taarifa za muhimu, inapendekezwa kutumia tafsiri ya mtafsiri mtaalamu wa kibinadamu. Hatuwajibiki kwa kutokuelewana au tafsiri isiyo sahihi inayotokana na matumizi ya tafsiri hii.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
